The objective of this mini-project is to define and train a Conditional VAE
(CVAE)

D'après IBM : 

L’un des inconvénients des VAE « vanilla » classiques est que l’utilisateur n’a aucun contrôle sur les sorties spécifiques générées par l’auto-encodeur. Par exemple, un VAE classique entraîné à partir du jeu de données MNIST mentionné précédemment générera de nouveaux échantillons de chiffres manuscrits compris entre 0 et 9, mais il ne peut pas être contraint de ne produire que des 4 et des 7.

Comme leur nom l'indique, les VAE conditionnelles (CVAE) permettent d'obtenir des sorties conditionnées par des entrées spécifiques, plutôt que de générer uniquement des variations de données d'entraînement au hasard. Pour ce faire, des éléments d'apprentissage supervisé (ou semi-supervisé) sont incorporés aux objectifs de formation traditionnellement non supervisés des auto-encodeurs conventionnels.

En entraînant le modèle sur des exemples étiquetés de variables spécifiques, ces variables peuvent être utilisées pour conditionner la sortie du décodeur. Par exemple, un CVAE peut d'abord être entraîné sur un grand jeu de données d'images faciales, puis en utilisant l'apprentissage supervisé pour apprendre un codage latent pour les « barbes » afin qu'il puisse produire de nouvelles images de visages barbus.


D'après https://medium.com/@mathparracho/variational-autoencoders-vaes-cvaes-%CE%B2-vaes-generative-intuition-practice-4b7b4011f55b

What is a Conditional Variational Autoencoder (CVAE)?

The CVAE (Conditional Variational Autoencoder) is a modification of the traditional VAE that introduces conditional outputs based on the input data.

Here we introduce a simple trick to make the network conditional to the input:
Press enter or click to view image in full size
Label Concatenation in the Channel Dimension

The idea is simply to concatenate the information we want to conditionalize in the channel dimension:

class ConditionalEncoder(nn.Module):
    def __init__(self, input_channels, hidden_dim, z_dim, num_classes):
        super(Encoder, self).__init__()
        self.num_classes = num_classes
        self.conv1 = nn.Conv2d(input_channels + num_classes, 32, kernel_size=4, stride=2, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1)
        self.fc_mu = nn.Linear(128 * 4 * 4, z_dim)
        self.fc_logvar = nn.Linear(128 * 4 * 4, z_dim)

    def forward(self, x, labels):

        # one-hot encode labels and expand dimensions shape [batch_size, num_classes, spatialY, spatialX],
        y = one_hot_encode(labels, self.num_classes).unsqueeze(2).unsqueeze(3) # [64, 10, 1, 1]
        y = y.expand(-1, -1, x.size(2), x.size(3)) #[64, 10, 28, 28] make a "layer of conditional information"
        # concatenate along the channel dimension
        x = torch.cat([x, y], dim=1)


        h = F.relu(self.conv1(x))
        h = F.relu(self.conv2(h))
        h = F.relu(self.conv3(h))
        h = h.view(h.size(0), -1)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

Bonus: We can find a similar approach to deal with spatial coordinates here in this paper with CoorConvs: Paper

So, for instance, if I want to generate only new “0” samples:
Press enter or click to view image in full size

Or, if I want to generate only new “1” samples:
Press enter or click to view image in full size

Or, only new “2” samples:
Press enter or click to view image in full size




In [3]:
%load_ext autoreload
%autoreload 2

import numpy as np
import torch
from torch.utils.data import DataLoader
import torchvision
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.nn.functional as F


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


https://medium.com/@sofeikov/implementing-conditional-variational-auto-encoders-cvae-from-scratch-29fcbb8cb08f

https://github.com/timbmg/VAE-CVAE-MNIST/blob/master/train.py

In [ ]:
from torchvision import datasets, transforms
batch_size = 128

# Data loading
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.FashionMNIST(root='data_cvae', train=True, transform=transform, download=False)
test_dataset = datasets.FashionMNIST(root='data_cvae', train=False, transform=transform, download=False)

train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

100.0%
100.0%
100.0%
100.0%


In [ ]:
class ConditionalVAE(nn.Module):
    def __init__(self, latent_dim=10):
        super(ConditionalVAE, self).__init__()
        self.latent_dim = latent_dim
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),  # Output: (32, 14, 14)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),  # Output: (64, 7, 7)
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),  # Output: (128, 4, 4)
            nn.BatchNorm2d(128),
            nn.ReLU(),
        )

        # Fully connected layers for mean and log variance
        self.fc_mu = nn.Linear(128 * 4 * 4, latent_dim) 
        self.fc_logvar = nn.Linear(128 * 4 * 4, latent_dim) 
        self.fc_decode = nn.Linear(latent_dim, 128 * 4 * 4) 
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1),  # Output: (64, 8, 8)
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),  # Output: (32, 16, 16)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1),  # Output: (1, 28, 28)
            nn.Sigmoid()
        )
    
    def encode(self, x):
        x = self.encoder(x)
        x = x.view(-1, 128 * 4 * 4) # Flatten the output of the convolutional layers
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        return mu, logvar
    

    def sample(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        x = self.fc_decode(z)
        x = x.view(-1, 128, 4, 4)  # Reshape to (128, 4, 4) for the decoder
        x = self.decoder(x)
        return x

    def forward(self, x, y):
        mu, logvar = self.encode(x)
        z = self.sample(mu, logvar)
        return self.decode(z), mu, logvar 
        
    # TODO: Implement the forward function
    # by combining the encode, sample and decode functions
